In [ ]:
import os, sys, json, shutil, numpy as np, subprocess, time
from pathlib import Path

COMP = Path('/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation')
WORK = Path('/kaggle/working')
sys.path.insert(0, str(COMP / 'kickmix'))
sys.path.insert(0, str(COMP / 'verifiers'))
sys.path.insert(0, '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit')
sys.path.insert(0, '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit/verifier')

SRC = Path('/kaggle/input/datasets/michaelrustad/vqcc-dfthc-r10-merged-groups/src')
sys.path.insert(0, str(SRC))

import types
dfthcsos = types.ModuleType('dfthcsos')
sys.modules['dfthcsos'] = dfthcsos
scoreinfra = types.ModuleType('dfthcsos.scoreinfra')
import dfthc_math as dm
for attr in dir(dm):
    if not attr.startswith('__'):
        setattr(scoreinfra, attr, getattr(dm, attr))
sys.modules['dfthcsos.scoreinfra'] = scoreinfra
dfthcsos.scoreinfra = scoreinfra

bp_members = Path('/kaggle/input/datasets/busyaprime/33-qcc-v8-members')
sub_dir = WORK / 'submission'
sub_dir.mkdir(exist_ok=True)
(sub_dir / 'dfthcsos').mkdir(exist_ok=True)

shutil.copy2(bp_members / 'invsqrt.kmb', sub_dir / 'invsqrt.kmb')
shutil.copy2(bp_members / 'dqi.kmb', sub_dir / 'dqi.kmb')
for p in bp_members.glob('dfthcsos__*.kmb'):
    shutil.copy2(p, sub_dir / 'dfthcsos' / p.name.replace('dfthcsos__', ''))
shutil.copy2(Path('/kaggle/input/datasets/michaelrustad/vqcc-dfthc-r10-merged-groups/fact/dfthc_r10_b129_c46_brot8.npz'), sub_dir / 'dfthcsos' / 'submission.npz')

print('Setup complete')


In [ ]:
IN = sub_dir / 'dfthcsos' / 'submission.npz'
RUNS = WORK / 'runs'
RUNS.mkdir(exist_ok=True)

print('Step 1: finish.py')
t0 = time.time()
result = subprocess.run(
    [sys.executable, str(SRC / 'finish.py'), str(IN), str(RUNS / 'fin.npz'), '--kappa', '0'],
    capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=str(SRC) + os.pathsep + str(COMP / 'kickmix') + os.pathsep + str(COMP / 'verifiers') + os.pathsep + '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit' + os.pathsep + '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit/verifier')
)
print(result.stdout[-500:] if result.stdout else 'NO STDOUT')
print(result.stderr[-500:] if result.stderr else 'NO STDERR')
print(f'  Time: {time.time()-t0:.0f}s')
print('Output exists:', (RUNS / 'fin.npz').exists())
